In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuración estética para los gráficos que haremos después
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")

# Definir la ruta del archivo
input_path = '../data/Engineered.pkl'

print("Cargando dataset...")
if os.path.exists(input_path):
    df = pd.read_pickle(input_path)
    print("✅ Dataset 'Engineered.pkl' cargado correctamente.")
    print(f"📊 Dimensión inicial: {df.shape[0]} registros y {df.shape[1]} variables.")
else:
    print(f"❌ Error: No se encontró el archivo en {input_path}")

# Verificamos los tipos de datos
display(df.info())

Cargando dataset...
✅ Dataset 'Engineered.pkl' cargado correctamente.
📊 Dimensión inicial: 119143 registros y 16 variables.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119143 entries, 0 to 119142
Data columns (total 16 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   order_id                        119143 non-null  object 
 1   is_delayed                      115722 non-null  float64
 2   actual_delivery_days            115722 non-null  float64
 3   estimated_delivery_margin_days  115722 non-null  float64
 4   purchase_month                  119143 non-null  int32  
 5   purchase_day_of_week            119143 non-null  int32  
 6   product_weight_g                118290 non-null  float64
 7   product_volume_cm3              118290 non-null  float64
 8   product_category_name_english   116576 non-null  object 
 9   customer_zip_code_prefix        119143 non-null  int64  
 10  seller_zip_code_

None

Revisar si se quieren poner medianas para los datos faltantes o si prefiere dejarse vacío

In [2]:
print("Iniciando proceso de limpieza de datos...")

# 1. Copia de seguridad para trabajar tranquilos
df_clean = df.copy()

# 2. ELIMINACIÓN ESTRICTA: Borrar filas donde NO conocemos el target
df_clean.dropna(subset=['is_delayed'], inplace=True)
print(f"📉 Filas tras eliminar target nulo: {df_clean.shape[0]}")

# 3. IMPUTACIÓN NUMÉRICA: Rellenar huecos con la mediana estadística
num_cols_to_impute = ['product_weight_g', 'product_volume_cm3', 'freight_value', 'price']

for col in num_cols_to_impute:
    if col in df_clean.columns:
        mediana_col = df_clean[col].median()
        # Rellenamos los NaN con la mediana de esa columna
        df_clean[col] = df_clean[col].fillna(mediana_col)

# 4. IMPUTACIÓN CATEGÓRICA: Rellenar textos nulos
if 'product_category_name_english' in df_clean.columns:
    df_clean['product_category_name_english'] = df_clean['product_category_name_english'].fillna('Unknown')

# 5. ELIMINACIÓN RESIDUAL: Si queda algún nulo en IDs u otras columnas menores, lo borramos
df_clean.dropna(inplace=True)

# 6. CORRECCIÓN DE TIPOS: El target debe ser un entero (0 o 1), no un decimal
df_clean['is_delayed'] = df_clean['is_delayed'].astype(int)

print(f"✅ Limpieza completada. Dimensión final: {df_clean.shape[0]} registros.")
print("-" * 40)
print("🔍 Verificación de Nulos (Deberían ser todos 0):")
print(df_clean.isnull().sum())

Iniciando proceso de limpieza de datos...
📉 Filas tras eliminar target nulo: 115722
✅ Limpieza completada. Dimensión final: 115722 registros.
----------------------------------------
🔍 Verificación de Nulos (Deberían ser todos 0):
order_id                          0
is_delayed                        0
actual_delivery_days              0
estimated_delivery_margin_days    0
purchase_month                    0
purchase_day_of_week              0
product_weight_g                  0
product_volume_cm3                0
product_category_name_english     0
customer_zip_code_prefix          0
seller_zip_code_prefix            0
is_same_state                     0
customer_state_num_pred           0
seller_state_num_pred             0
freight_value                     0
price                             0
dtype: int64


In [3]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("Iniciando filtrado suave de outliers mediante K-Means Multivariado...")

# 1. Seleccionar solo las variables físicas/económicas continuas
# No usamos IDs, ni variables categóricas, ni el target
cols_outliers = ['product_weight_g', 'product_volume_cm3', 'freight_value', 'price']

# 2. Escalar los datos (VITAL)
# Si no escalamos, el volumen (que puede ser 50,000) anulará al precio (que puede ser 50)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[cols_outliers])

# 3. Entrenar K-Means
# Usamos k=5 para encontrar 5 "perfiles típicos" de envíos
kmeans = KMeans(n_clusters=5, random_state=42, n_init='auto')
clusters = kmeans.fit_predict(X_scaled)

# 4. Medir distancias a los centroides
# .transform() devuelve la distancia de cada punto a todos los centroides.
# Tomamos la distancia mínima (la distancia a SU cluster)
distancias = kmeans.transform(X_scaled)
df_clean['distancia_centroide'] = np.min(distancias, axis=1)

# 5. Definir el umbral "Suave" (Percentil 99)
# Solo eliminaremos el 1% de los pedidos que sean absolutamente bizarros
umbral_corte = np.percentile(df_clean['distancia_centroide'], 99)
print(f"Umbral de corte de distancia (Percentil 99): {umbral_corte:.2f}")

# 6. Ejecutar el filtrado
# Nos quedamos con los que están POR DEBAJO del umbral
outliers_detectados = df_clean[df_clean['distancia_centroide'] > umbral_corte].shape[0]
df_filtered = df_clean[df_clean['distancia_centroide'] <= umbral_corte].copy()

# Limpiar las columnas temporales que usamos para el cálculo
df_filtered.drop(columns=['distancia_centroide'], inplace=True)

print(f"🚨 Outliers eliminados: {outliers_detectados} registros extremos.")
print(f"✅ Dimensión final tras filtrado K-Means: {df_filtered.shape[0]} registros.")

# Opcional: Echar un vistazo a lo que acabamos de borrar para validar que eran raros
display(df_clean[df_clean['distancia_centroide'] > umbral_corte][cols_outliers].head())

Iniciando filtrado suave de outliers mediante K-Means Multivariado...
Umbral de corte de distancia (Percentil 99): 4.03
🚨 Outliers eliminados: 1155 registros extremos.
✅ Dimensión final tras filtrado K-Means: 114567 registros.


,product_weight_g,product_volume_cm3,freight_value,price
20,20850.0,125000.0,77.45,1299.00
362,544.0,5148.0,129.84,217.85
365,544.0,5148.0,129.84,217.85
393,18450.0,59976.0,107.43,1148.00
411,550.0,2816.0,32.09,1999.00


In [4]:
import os

# Asegurarnos de que la carpeta existe (por si acaso)
os.makedirs('../data', exist_ok=True)

# Definir las rutas (he corregido el nombre a 'Filtered', cámbialo si el error era intencional)
ruta_csv = '../data/Filtered.csv'
ruta_pkl = '../data/Filtered.pkl'

# 1. Guardar como CSV (Formato universal y ligero para tabular)
df_filtered.to_csv(ruta_csv, index=False)
print(f"✅ Dataset maestro guardado en formato CSV: {ruta_csv}")

# 2. Guardar como Pickle (Opcional, pero muy recomendado para no perder los tipos de datos 'int' de tus IDs)
df_filtered.to_pickle(ruta_pkl)
print(f"✅ Dataset maestro guardado en formato Pickle: {ruta_pkl}")

# Comprobación de tamaño
tamano_mb = os.path.getsize(ruta_csv) / (1024 * 1024)
print(f"📦 Tamaño del archivo CSV: {tamano_mb:.2f} MB")

✅ Dataset maestro guardado en formato CSV: ../data/Filtered.csv
✅ Dataset maestro guardado en formato Pickle: ../data/Filtered.pkl
📦 Tamaño del archivo CSV: 11.85 MB
